In [ ]:
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

In [ ]:
spark = create_spark_session(
    aws_profile="default"
)

In [ ]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
ev.configurations.to_sdf().show()

In [ ]:
ev.configurations.add([
    teehr.Configuration(
        name="psu_dhbv_hf", 
        timeseries_type="secondary", 
        description="PSU differentiable HBV simulations, AORC forcing, Hydrofabric v2.2, 1980-1995 training period"
    ),
    teehr.Configuration(
        name="psu_dprms_hf", 
        timeseries_type="secondary", 
        description="PSU differentiable PRMS simulations, AORC forcing, Hydrofabric v2.2, 1980-1995 training period"
    ),
    teehr.Configuration(
        name="psu_dsacsma_hf", 
        timeseries_type="secondary", 
        description="PSU differentiable SACSMA simulations, AORC forcing, Hydrofabric v2.2, 1980-1995 training period"
    ),
    teehr.Configuration(
        name="psu_densemble_hf", 
        timeseries_type="secondary", 
        description="PSU ensemble mean of dHBV-dPRMS-dSACSMA, AORC forcing, Hydrofabric v2.2, 1980-1995 training period"
    )
])

First lets check validation manually to see if the data passes.

In [ ]:
# Loading local copies of the data.  Originals:
# s3://ciroh-rti-public-data/teehr-data-warehouse/common/simulations/psu_differentiable_simulations/teehr_format/*.parquet
sdf = spark.read.parquet(
        "/data/playground/mdenno/psu_differentiable_simulations/teehr_format/psu_dhbv_hf.parquet",
        "/data/playground/mdenno/psu_differentiable_simulations/teehr_format/psu_dprms_hf.parquet",
        "/data/playground/mdenno/psu_differentiable_simulations/teehr_format/psu_dsacsma_hf.parquet",
        "/data/playground/mdenno/psu_differentiable_simulations/teehr_format/psu_densemble_hf.parquet"
)

In [ ]:
sdf.show()

In [ ]:
sdf.count()

In [ ]:
ev._validate.dataframe(
    sdf,
    table_schema=ev.secondary_timeseries.schema_func(),
    uniqueness_fields=ev.secondary_timeseries.uniqueness_fields,
    foreign_keys=ev.secondary_timeseries.foreign_keys
)

Ok, now let's load the data.

In [ ]:
ev.secondary_timeseries.load_dataframe(sdf)

In [ ]:
ev.secondary_timeseries.filter("configuration_name = 'psu_dhbv_hf'").to_sdf().count()

In [ ]:
ev.secondary_timeseries.filter("configuration_name = 'psu_dprms_hf'").to_sdf().count()

In [ ]:
ev.secondary_timeseries.filter("configuration_name = 'psu_dsacsma_hf'").to_sdf().count()

In [ ]:
ev.secondary_timeseries.filter("configuration_name = 'psu_densemble_hf'").to_sdf().count()

In [ ]:
spark.stop()